# Online Model Calibration (Joint State-Parameter Estimation)

Welcome to the eighth interactive notebook of the `digital-twins` repository.

Let's see a realistic problem: What if the parameters of your simulation model are unknown, or worse, what if they change dynamically during the operation of the real system?

We cannot simply run our simulation with wrong parameters. We must use real-time sensor data to **calibrate the model online** while simultaneously estimating the state. We do this by defining an **Augmented State Vector ($z_k$)** that glues the physical state ($x_k$) and the unknown parameter ($\theta_k$) together:

$$ z_k = \begin{bmatrix} x_k \\ \theta_k \end{bmatrix} $$

### The Trick: Parameter Perturbation (Random Walk)
If a parameter is "static" in our model, all particles will just keep whatever parameter value they were initialized with. To allow the Particle Filter to search for the *true* parameter value and track it if it changes over time, we must inject artificial noise (a random walk) into the parameter during the Prediction step:

$$ \theta_k = \theta_{k-1} + \zeta_k, \quad \zeta_k \sim N(0, W_k) $$

Where $W_k$ is the variance of our parameter perturbation. If $W_k = 0$, the filter cannot adapt. If $W_k$ is too high, the estimated parameter bounces around wildly.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
from ipywidgets import interact, FloatSlider

# Ensure our plotting styles are clean
import warnings
warnings.filterwarnings('ignore')

### Interactive Learning: The Drone and the Sudden Gust of Wind

**The Scenario:** A drone is flying in a 1D straight line. We track its position using a noisy GPS. The drone is fighting a headwind. The headwind is an **unknown parameter ($\theta$)** in our physics model.
- At Time Step 0 to 20, the true wind is pushing the drone at `5.0 m/s`.
- At Time Step 20, a sudden gust hits, and the true wind shifts to `12.0 m/s`.

**Instructions:**
Adjust the `parameter_noise_std` ($\sqrt{W_k}$) slider to see how the Particle Filter tracks the unknown wind parameter.
1. **Set it to `0.0`**: The filter makes an initial guess for the wind and locks in. When the gust hits at $T=20$, the filter's model is suddenly wrong, and the entire position tracking system fails entirely!
2. **Set it to `3.0`**: The filter adapts instantly to the wind gust, but the parameter estimate is incredibly noisy and chaotic.
3. **Find the Sweet Spot**: Find the ideal perturbation variance that allows the filter to "learn" the new parameter smoothly without overfitting to the sensor noise.

In [ ]:
def interactive_joint_estimation(parameter_noise_std=0.5):
    # --- Fixed Seed for Reproducible True Reality ---
    np.random.seed(42)
    
    # --- 1. System Parameters ---
    total_steps = 40
    dt = 1.0
    sensor_noise_std = 4.0
    process_noise_std = 1.0
    N = 1000 # Number of particles
    
    # --- 2. Generate the True World (Hidden from Filter) ---
    true_pos = np.zeros(total_steps)
    true_wind = np.zeros(total_steps)
    measurements = np.zeros(total_steps)
    
    current_pos = 0.0
    for k in range(total_steps):
        # The dynamic parameter: Wind jumps from 5.0 to 12.0 at k=20
        current_wind = 5.0 if k < 20 else 12.0
        
        current_pos += current_wind * dt + np.random.normal(0, process_noise_std)
        
        true_pos[k] = current_pos
        true_wind[k] = current_wind
        measurements[k] = current_pos + np.random.normal(0, sensor_noise_std)
        
    # --- 3. Joint State-Parameter Particle Filter ---
    # State Vector z_k = [position, wind_speed]
    # We initialize with terrible guesses: Wind is randomly between 0 and 20.
    particles = np.column_stack((
        np.random.normal(0, 5.0, N),         # Initial Pos
        np.random.uniform(0.0, 20.0, N)      # Initial Wind (Parameter)
    ))
    weights = np.ones(N) / N
    
    est_pos = np.zeros(total_steps)
    est_wind = np.zeros(total_steps)
    
    for k in range(total_steps):
        # A. PREDICT (State Transition + Parameter Perturbation)
        # Equation 6.13: θ_k = θ_{k-1} + ζ_k
        particles[:, 1] += np.random.normal(0, parameter_noise_std, N)
        
        # Equation 6.5: x_k = f(x_{k-1}, θ_k)
        particles[:, 0] += particles[:, 1] * dt + np.random.normal(0, process_noise_std, N)
        
        # B. UPDATE (Likelihood)
        variance = sensor_noise_std ** 2
        diff = particles[:, 0] - measurements[k]
        likelihoods = np.exp(-0.5 * (diff ** 2) / variance)
        
        weights *= likelihoods
        weight_sum = np.sum(weights)
        if weight_sum == 0:
            weights = np.ones(N) / N
        else:
            weights /= weight_sum
            
        # C. ESTIMATE (Weighted Mean)
        est_pos[k] = np.average(particles[:, 0], weights=weights)
        est_wind[k] = np.average(particles[:, 1], weights=weights)
        
        # D. RESAMPLE (Systematic Resampling)
        cum_sum = np.cumsum(weights)
        cum_sum[-1] = 1.0
        indices = np.zeros(N, dtype=int)
        r = np.random.uniform(0, 1.0 / N)
        i = 0
        for j in range(N):
            u = r + j / N
            while u > cum_sum[i]:
                i += 1
            indices[j] = i
            
        particles = particles[indices]
        weights = np.ones(N) / N
        
    # --- 4. Plotting ---
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 9), sharex=True)
    
    time_arr = np.arange(total_steps)
    
    # Top Plot: Parameter Tracking (Wind)
    ax1.plot(time_arr, true_wind, 'k--', linewidth=2, label='True Parameter (Wind Gust)')
    ax1.plot(time_arr, est_wind, 'm-o', linewidth=2, label=f'PF Estimated Parameter ($\sigma_W$={parameter_noise_std})')
    ax1.axvline(20, color='gray', linestyle=':', alpha=0.7)
    ax1.set_title("Online Model Calibration: Estimating the Unknown Wind Parameter", fontweight='bold')
    ax1.set_ylabel("Wind Speed Parameter (m/s)")
    ax1.legend(loc='lower right')
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim(0, 15)
    
    # Bottom Plot: State Tracking (Position)
    ax2.plot(time_arr, true_pos, 'k--', linewidth=2, label='True Position')
    ax2.plot(time_arr, measurements, 'rx', alpha=0.5, label='Noisy GPS Measurements')
    ax2.plot(time_arr, est_pos, 'b-', linewidth=2, label='PF Estimated Position')
    ax2.axvline(20, color='gray', linestyle=':', alpha=0.7, label='Gust Hits Here')
    ax2.set_title("Dynamic State Estimation: Tracking Drone Position", fontweight='bold')
    ax2.set_ylabel("Position (m)")
    ax2.set_xlabel("Time Step")
    ax2.legend(loc='upper left')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Create the interactive widget
interact(interactive_joint_estimation, 
         parameter_noise_std=FloatSlider(value=0.5, min=0.0, max=3.0, step=0.1, description='Parameter Noise Std ($\zeta$):', layout={'width':'500px'}));